# 01 - Google Trends Ingestion

## Purpose
This notebook will be used for Google Trends extraction for the Coffee Dashboard project.

## Current status
Paused for now, as the project currently used a saved CSV file as input for Power BI development.

---

In [117]:
import pandas as pd
from pathlib import Path

In [118]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Interim data folder:", INTERIM_DIR)

Project root: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard
Raw data folder: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\raw
Interim data folder: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim


In [119]:
RAW_FILE = RAW_DIR / "google_trends_expanded.csv"

print("Raw file exists:", RAW_FILE.exists())
print("Raw file path:", RAW_FILE)

Raw file exists: True
Raw file path: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\raw\google_trends_expanded.csv


In [120]:
df_raw = pd.read_csv(RAW_FILE)
df_raw.head()

,date,espresso,cappuccino,latte,americano,iced coffee,iced latte,cold brew,frappuccino,flat white,pumpkin spice latte,peppermint mocha,gingerbread latte,caramel macchiato,dalgona coffee,mushroom coffee
0,2016-04-01,26,8,34,43,4,1,1,3,2,0,0,0,2,0,0
1,2016-05-01,24,9,30,44,5,1,2,5,2,0,0,0,2,0,0
2,2016-06-01,25,8,30,43,6,1,3,3,2,0,0,0,2,0,0
3,2016-07-01,25,8,29,41,6,1,3,4,2,0,0,0,2,0,0
4,2016-08-01,24,8,29,43,5,1,3,3,1,1,0,0,2,0,0


In [121]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   date                 121 non-null    str  
 1   espresso             121 non-null    int64
 2   cappuccino           121 non-null    int64
 3   latte                121 non-null    int64
 4   americano            121 non-null    int64
 5   iced coffee          121 non-null    int64
 6   iced latte           121 non-null    int64
 7   cold brew            121 non-null    int64
 8   frappuccino          121 non-null    int64
 9   flat white           121 non-null    int64
 10  pumpkin spice latte  121 non-null    int64
 11  peppermint mocha     121 non-null    int64
 12  gingerbread latte    121 non-null    int64
 13  caramel macchiato    121 non-null    int64
 14  dalgona coffee       121 non-null    int64
 15  mushroom coffee      121 non-null    int64
dtypes: int64(15), str(1)
memory usage: 15

In [122]:
df_raw.columns.tolist()

['date',
 'espresso',
 'cappuccino',
 'latte',
 'americano',
 'iced coffee',
 'iced latte',
 'cold brew',
 'frappuccino',
 'flat white',
 'pumpkin spice latte',
 'peppermint mocha',
 'gingerbread latte',
 'caramel macchiato',
 'dalgona coffee',
 'mushroom coffee']

---

In [123]:
df_fact = df_raw.melt(
    id_vars="date",
    var_name="keyword",
    value_name="search_interest"
)

df_fact["date"] = pd.to_datetime(df_fact["date"])
df_fact["search_interest"] = pd.to_numeric(df_fact["search_interest"])

df_fact = df_fact.sort_values(["keyword", "date"]).reset_index(drop=True)

df_fact.head

<bound method NDFrame.head of            date              keyword  search_interest
0    2016-04-01            americano               43
1    2016-05-01            americano               44
2    2016-06-01            americano               43
3    2016-07-01            americano               41
4    2016-08-01            americano               43
...         ...                  ...              ...
1810 2025-12-01  pumpkin spice latte                0
1811 2026-01-01  pumpkin spice latte                0
1812 2026-02-01  pumpkin spice latte                0
1813 2026-03-01  pumpkin spice latte                0
1814 2026-04-01  pumpkin spice latte                0

[1815 rows x 3 columns]>

In [124]:
df_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 1815 entries, 0 to 1814
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             1815 non-null   datetime64[us]
 1   keyword          1815 non-null   str           
 2   search_interest  1815 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 42.7 KB


In [125]:
OUTPUT_FILE = INTERIM_DIR / "google_trends_fact.csv"
df_fact.to_csv(OUTPUT_FILE, index=False)

print(f"Saved fact table to: {OUTPUT_FILE}")

Saved fact table to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\google_trends_fact.csv


In [126]:
# Run checks on the fact table

# Basic structure
print("Fact table shape:", df_fact.shape)
print("\nColumns:")
print(df_fact.columns.tolist())

# Null checks
print("\nNull values by column:")
print(df_fact.isna().sum())

# Duplicate checks
duplicate_count = df_fact.duplicated().sum()
print("\nNumber of duplicated rows:", duplicate_count)

# Check keyword list
print("\nKeywords:")
print(sorted(df_fact["keyword"].dropna().unique()))

# Check date range
print("\nMin date:", df_fact["date"].min())
print("Max date:", df_fact["date"].max())

# Search interest range
print("\nMinimum search interest:", df_fact["search_interest"].min())
print("Maximum search interest:", df_fact["search_interest"].max())

Fact table shape: (1815, 3)

Columns:
['date', 'keyword', 'search_interest']

Null values by column:
date               0
keyword            0
search_interest    0
dtype: int64

Number of duplicated rows: 0

Keywords:
['americano', 'cappuccino', 'caramel macchiato', 'cold brew', 'dalgona coffee', 'espresso', 'flat white', 'frappuccino', 'gingerbread latte', 'iced coffee', 'iced latte', 'latte', 'mushroom coffee', 'peppermint mocha', 'pumpkin spice latte']

Min date: 2016-04-01 00:00:00
Max date: 2026-04-01 00:00:00

Minimum search interest: 0
Maximum search interest: 100


---

In [127]:
# Create the date dimension table

min_date = df_fact["date"].min()
max_date = df_fact["date"].max()

date_range = pd.date_range(start=min_date, end=max_date, freq="D")

dim_date = pd.DataFrame({"date": date_range})

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%B")
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["week_of_year"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_of_month"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.strftime("%A")

dim_date.head()

,date,year,quarter,month_number,month_name,year_month,week_of_year,day_of_month,day_name
0,2016-04-01,2016,Q2,4,April,2016-04,13,1,Friday
1,2016-04-02,2016,Q2,4,April,2016-04,13,2,Saturday
2,2016-04-03,2016,Q2,4,April,2016-04,13,3,Sunday
3,2016-04-04,2016,Q2,4,April,2016-04,14,4,Monday
4,2016-04-05,2016,Q2,4,April,2016-04,14,5,Tuesday


In [128]:
dim_date.info()

<class 'pandas.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          3653 non-null   datetime64[us]
 1   year          3653 non-null   int32         
 2   quarter       3653 non-null   str           
 3   month_number  3653 non-null   int32         
 4   month_name    3653 non-null   str           
 5   year_month    3653 non-null   str           
 6   week_of_year  3653 non-null   int64         
 7   day_of_month  3653 non-null   int32         
 8   day_name      3653 non-null   str           
dtypes: datetime64[us](1), int32(3), int64(1), str(4)
memory usage: 214.2 KB


In [129]:
DATE_DIM_FILE = INTERIM_DIR / "dim_date.csv"
dim_date.to_csv(DATE_DIM_FILE, index=False)

print(f"Saved date dimension to: {DATE_DIM_FILE}")
print("Shape:", dim_date.shape)

Saved date dimension to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\dim_date.csv
Shape: (3653, 9)


---

In [130]:
# Create the trend dimension table (to improve)

dim_trend = pd.DataFrame({
    "keyword": df_fact["keyword"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

# Mapping
trend_metadata = {
    # Core hot coffee drinks
    "espresso": {"category": "coffee", "style": "espresso", "serving_style": "hot", "trend_type": "core", "contains_milk": False},
    "americano": {"category": "coffee", "style": "americano", "serving_style": "hot", "trend_type": "core", "contains_milk": False},
    "cappuccino": {"category": "coffee", "style": "cappuccino", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "flat white": {"category": "coffee", "style": "flat_white", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "mocha": {"category": "coffee", "style": "mocha", "serving_style": "hot", "trend_type": "core", "contains_milk": True},

    # Core cold coffee drinks
    "iced coffee": {"category": "coffee", "style": "iced_coffee", "serving_style": "cold", "trend_type": "core", "contains_milk": False},
    "cold brew": {"category": "coffee", "style": "cold_brew", "serving_style": "cold", "trend_type": "core", "contains_milk": False},
    "iced latte": {"category": "coffee", "style": "latte", "serving_style": "cold", "trend_type": "core", "contains_milk": True},
    "iced americano": {"category": "coffee", "style": "americano", "serving_style": "cold", "trend_type": "core", "contains_milk": False},

    # Flavoured lattes
    "vanilla latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "caramel latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "hazelnut latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},

    # Seasonal or limited-time offerings
    "pumpkin spice latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "seasonal", "contains_milk": True},
    "peppermint mocha": {"category": "coffee", "style": "mocha", "serving_style": "hot", "trend_type": "seasonal", "contains_milk": True},
    "gingerbread latte": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "seasonal", "contains_milk": True},

    # Frappuccinos / desserts
    "frappuccino": {"category": "coffee", "style": "frappuccino", "serving_style": "cold", "trend_type": "core", "contains_milk": True},
    "caramel frappuccino": {"category": "coffee", "style": "frappuccino", "serving_style": "cold", "trend_type": "core", "contains_milk": True},
    "mocha frappuccino": {"category": "coffee", "style": "frappuccino", "serving_style": "cold", "trend_type": "core", "contains_milk": True},

    # Macchiato drinks
    "caramel macchiato": {"category": "coffee", "style": "latte", "serving_style": "hot", "trend_type": "core", "contains_milk": True},
    "iced caramel macchiato": {"category": "coffee", "style": "latte", "serving_style": "cold", "trend_type": "core", "contains_milk": True},

    # Niche or emerging trends
    "dalgona coffee": {"category": "coffee", "style": "whipped_coffee", "serving_style": "cold", "trend_type": "trendy", "contains_milk": False},
    "mushroom coffee": {"category": "coffee", "style": "functional_coffee", "serving_style": "hot", "trend_type": "niche", "contains_milk": False},

    # Other brewing methods
    "pour over coffee": {"category": "coffee", "style": "pour_over", "serving_style": "hot", "trend_type": "niche", "contains_milk": False},
    "french press coffee": {"category": "coffee", "style": "french_press", "serving_style": "hot", "trend_type": "niche", "contains_milk": False},
    "drip coffee": {"category": "coffee", "style": "drip_coffee", "serving_style": "hot", "trend_type": "core", "contains_milk": False},
}

dim_trend["category"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("category"))
dim_trend["style"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("style"))
dim_trend["serving_style"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("serving_style"))
dim_trend["trend_type"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("trend_type"))
dim_trend["contains_milk"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("contains_milk"))

dim_trend.head()

,keyword,category,style,serving_style,trend_type,contains_milk
0,americano,coffee,americano,hot,core,False
1,cappuccino,coffee,cappuccino,hot,core,True
2,caramel macchiato,coffee,latte,hot,core,True
3,cold brew,coffee,cold_brew,cold,core,False
4,dalgona coffee,coffee,whipped_coffee,cold,trendy,False


In [131]:
TREND_DIM_FILE = INTERIM_DIR / "dim_trend.csv"
dim_trend.to_csv(TREND_DIM_FILE, index=False)

print(f"Saved trend dimension to: {TREND_DIM_FILE}")

Saved trend dimension to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\dim_trend.csv
